# Walking skeleton — `bc_planning_optimizer.run`

End-to-end tracer bullet for the planning-optimizer math seam (issue #11, [ADR 0009](../../docs/adr/0009-python-package-with-api-and-file-seam.md)).

Reads a synthetic 5-row `(item_no, variant_code, location_code, daily_demand, lead_time_days)` extract and emits a recommendations JSON using deliberately-naive math:

```
reorder_point = mean(daily_demand) × mean(lead_time_days)
safety_stock  = reorder_point / 2
```

No policy change. No bootstrap LTD. No simulator. Real math lands in later slices.

In [1]:
import json
import shutil
import tempfile
from pathlib import Path

import pandas as pd

from bc_planning_optimizer import run

NOTEBOOK_DIR = Path.cwd()
FIXTURE = NOTEBOOK_DIR / ".." / "tests" / "fixtures" / "synthetic_extract.csv"
FIXTURE = FIXTURE.resolve()
FIXTURE

PosixPath('/home/fbakkensen/repos/bc-on-linux-test/planning-optimizer/tests/fixtures/synthetic_extract.csv')

## Synthetic extract

In [2]:
pd.read_csv(FIXTURE, keep_default_na=False)

,item_no,variant_code,location_code,daily_demand,lead_time_days
0,ITEM-A,,BLUE,10,5
1,ITEM-A,,BLUE,20,5
2,ITEM-A,,BLUE,30,7
3,ITEM-A,,BLUE,40,6
4,ITEM-A,,BLUE,50,7


## Run the pipeline

`run()` writes `recommendations.json` next to its input. Copy the fixture into a tempdir so the notebook does not write into the repo.

In [3]:
workdir = Path(tempfile.mkdtemp(prefix="planning-optimizer-skeleton-"))
extract_copy = workdir / "synthetic_extract.csv"
shutil.copy(FIXTURE, extract_copy)

output_path = run(extract_copy)
output_path

PosixPath('/tmp/planning-optimizer-skeleton-rmp33ch9/recommendations.json')

## Recommendations

In [4]:
print(json.dumps(json.loads(output_path.read_text()), indent=2))

{
  "recommendations": [
    {
      "item_no": "ITEM-A",
      "variant_code": "",
      "location_code": "BLUE",
      "reorder_point": 180.0,
      "safety_stock": 90.0
    }
  ]
}
